In [1]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

In [2]:
dataset = load_dataset("yelp_review_full")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})


In [3]:
train_dataset = dataset['train']
test_dataset = dataset['test']

restaurant_train_review = train_dataset.filter(
    lambda x: 'restaurant' in x['text'].lower()
)

restaurant_test_review = test_dataset.filter(
    lambda x : 'restaurant' in x['text'].lower()
)

number_of_reviews = 5000
subset_train_reviews = restaurant_train_review.shuffle(seed=42).select(range(number_of_reviews))
subset_test_reviews = restaurant_test_review.shuffle(seed=42).select(range(number_of_reviews))

subset_dataset = {
    'train':subset_train_reviews,
    'test':subset_test_reviews
}

yelp_restaurant_dataset = DatasetDict(subset_dataset)
print(yelp_restaurant_dataset)

Filter:   0%|          | 0/650000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 5000
    })
})


In [4]:
model_checkpoint = 'distilbert-base-uncased'
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=5)
#

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
if torch.cuda.is_available():
  device = torch.device('cuda')
elif torch.backends.mps.is_available():
  device = torch.device('mps')
else:
  device = torch.device('cpu')

model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenizer_fun(examples):
  return tokenizer(
      examples['text'],
      padding='max_length',
      truncation=True,
      max_length=512,
  )

tokenized_dataset = yelp_restaurant_dataset.map(
    tokenizer_fun,
    batched=True
)

tokenized_dataset

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
})

In [7]:
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy ='epoch',
    save_strategy ='epoch',
    learning_rate = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 3,
    weight_decay = 0.01,
    logging_dir = './logs',
    logging_steps = 10,
    save_steps=500,
    load_best_model_at_end = True,
)

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test']
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,1.079078,0.983015
2,0.843855,0.933175
3,0.761903,0.928532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=939, training_loss=0.9480823018172552, metrics={'train_runtime': 980.3106, 'train_samples_per_second': 15.301, 'train_steps_per_second': 0.958, 'total_flos': 1987117286400000.0, 'train_loss': 0.9480823018172552, 'epoch': 3.0})

In [19]:
model.save_pretrained("./results/final_model_multiclass")
tokenizer.save_pretrained("./results/final_tokenizer_multiclass")
eval_results = trainer.evaluate()
print(eval_results)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.9285321235656738, 'eval_runtime': 70.3802, 'eval_samples_per_second': 71.043, 'eval_steps_per_second': 4.447, 'epoch': 3.0}


In [14]:
new_review = [
 "The food was amazing and the service was excellent!",
 "The restaurant was dirty and the food was cold.",
 "Decent experience, but nothing special."
]

new_tokenizer =AutoTokenizer.from_pretrained("./results/final_tokenizer_multiclass")
new_model = AutoModelForSequenceClassification.from_pretrained("./results/final_model_multiclass")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [20]:
inputs = new_tokenizer(new_review,
                       padding='max_length',
                       return_tensors='pt',
                       truncation=True)

inputs = {key:value.to(device) for key, value in inputs.items()}

In [21]:
new_model = AutoModelForSequenceClassification.from_pretrained("./results/final_model_multiclass")
new_model.to(device)

new_model.eval()

with torch.no_grad():
  outputs = new_model(**inputs)
  logits = outputs.logits
  predictions = torch.argmax(logits, dim=-1)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [22]:
star_rating = predictions + 1
for review, rating in zip(new_review, star_rating):
  print(f"Review: {review}\nPredicted Star Rating: {rating.item()}\n")

Review: The food was amazing and the service was excellent!
Predicted Star Rating: 5

Review: The restaurant was dirty and the food was cold.
Predicted Star Rating: 1

Review: Decent experience, but nothing special.
Predicted Star Rating: 2

